# From a Lean proof state to a top-10 pool — two systems, one final pick

**Goal of this notebook.** Take a Lean 4 proof state, turn it into a graph, run the published
graph neural network to rank the 228 candidate tactic *families*, keep its **top-10** as the
candidate pool, and compare the two systems that both narrow that pool down to five and then
to one.

### Protocol

```
stage 1   GNN ranks all 228 tactics  ──►  candidate pool = top-10

stage 2a  System A — pure GNN    keeps the GNN's ranking of the ten
                                 ──►  advances its top-5  ──►  final pick = #1

stage 2b  System B — GNN + Laya  Laya re-ranks the same ten
                                 ──►  advances its top-5  ──►  final pick = #1

stage 3   the top-5 and the top-1 of both systems are evaluated
```

| system | how it orders the pool |
|---|---|
| **A — pure GNN** | the GNN's own probabilities; ranks 1–10 unchanged |
| **B — GNN + Laya** | Laya's `choice` probabilities over the same ten |

Top-5 and top-1 are two cut-offs of one ranking: a system advances its first five candidates,
and the final pick is simply the head of those five. Both systems work on the *same* ten
candidates, so `recall@10` is the shared pool ceiling and is identical for both. The headline
numbers are **`recall@5`** (did the target survive into the advanced five?) and **`recall@1`**
(was it the final pick?). The `cond@` columns restrict everything to rows whose target was in
the pool in the first place.

Everything below is **self-contained**: the small amount of graph/model code the run needs is
vendored into `atp_core.py` by a cell in this notebook, so no repository clone is required.

### Pipeline

```
HF dataset  test split ──► text_state ──► proof_state_to_dag ──► dag_to_pyg ──┐
                                                                            ├─► GATv2 GNN ─► p_gnn
HF model   pointer-gat-gru ─► backbone weights + node/tactic vocab ──────────┘        │
                                                                                      ▼
                                                                    candidate pool = GNN top-10
                                                                                      │
                                             ┌────────────────────────────────────────┴──────────┐
                                             ▼                                                   ▼
                                   System A — pure GNN                              System B — GNN + Laya
                                   GNN ranking of the ten                           Laya choice(q) on the ten
                                             │                                                   │
                                   advance top-5 ──► #1                          advance top-5 ──► #1
                                             │                                                   │
                                             └─────────────── evaluation ────────────────────────┘
```

### What you get at the end

* per-row top-10 ranking for each of the two systems,
* `recall@5` (the advanced five), `recall@1` (the final pick), `recall@10` (shared pool
  ceiling), conditional accuracy, MRR@10 and top-1/top-5 transitions,
* four figures, a CSV and a Markdown summary.

### Runtime expectations (Colab GPU T4)

| stage | time |
|---|---|
| install + downloads (~45 MB model, ~5 MB data, ~1.7 GB Laya) | 2–5 min |
| graph build for 500 rows | ~1 min |
| GNN forward passes | seconds |
| Laya re-rank, 500 calls (one per row over the ten) | ~1 min + model download |

### One verified finding worth reading before you run

The published bundle's `node_vocab.json` contains **no** S-expression labels at all
(`:app`, `:arg`, `:forall`, `FV0`, `HypRole:*`, `ArgPosition:*` are all absent) and is full of
**text-parser** labels (`App`, `Arrow`, `Forall`, `Explicit`, `State`, `Goal`, `Hyp`).
Building graphs the S-expression way therefore maps ~74 % of nodes to `<UNK>` and scores
**top-1 = 0.156**; building them the *text* way maps ~1.5 % to `<UNK>` and reproduces the
published **top-1 ≈ 0.315**. This notebook uses the text path, and there is a dedicated cell
that re-checks the vocabulary evidence.


In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# --- reproducibility -------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- artefacts -------------------------------------------------------------
MODEL_REPO = "jajostrains/Mathlib-Sexpr-GNN"
DATASET_REPO = "jajostrains/Mathlib-Normalized-Sexpr"
BUNDLE_NAME = "pointer-gat-gru"   # the GNN head used for tactic ranking
SPLIT = "test"

# --- evaluation ------------------------------------------------------------
N_ROWS = 500        # rows to evaluate; reduce this if you are short on time
TOP_K = (10, 5, 1)  # cut-offs reported for both systems
BATCH_SIZE = 64

# --- selection protocol ----------------------------------------------------
# stage 1: the GNN ranks every tactic; its top-10 becomes the candidate pool.
# stage 2: two systems order that same pool —
#            A: the GNN's own ranking, unchanged;
#            B: Laya's choice probabilities over the ten.
#          each system advances its top-5, and the head of those five is the
#          final pick (two cut-offs of one ranking).
# stage 3: recall@5 / recall@1 of both systems are what gets compared.
POOL_K = 10         # size of the candidate pool both systems work on
SELECT_K = 5        # the five each system advances; its head is the final pick

# --- re-ranking ------------------------------------------------------------
LAYA_MODEL = "convaiinnovations/laya"  # English, ModernBERT-large, 512-token context
LAYA_INSTRUCTIONS = "Which Lean 4 tactic family best matches this proof step?"
# One Laya call per row, over the GNN's top-10 only. A `choice` question spends one
# shared head_max_len budget (192 tokens) on all of its options and *refuses* the
# question when they no longer fit -- about 20 options with a one-line gloss -- so
# Laya cannot be asked to choose from the whole tactic vocabulary.
LAYA_K = POOL_K

# --- paths -----------------------------------------------------------------
WORK = Path("/content") if Path("/content").is_dir() else Path.cwd()
MODEL_DIR = WORK / "weights" / MODEL_REPO.replace("/", "--")
BUNDLE_DIR = MODEL_DIR / BUNDLE_NAME
DATASET_DIR = WORK / "data" / DATASET_REPO.replace("/", "--")
CACHE_DIR = WORK / "cache"
OUT_DIR = WORK / "outputs"
for _path in (MODEL_DIR, DATASET_DIR, CACHE_DIR, OUT_DIR):
    _path.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LAYA_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"work={WORK}")
print(f"device={DEVICE}  rows={N_ROWS}  seed={SEED}")
print(f"pool=top-{POOL_K}   advance=top-{SELECT_K}   cut-offs={TOP_K}")
print("systems: A = pure GNN ranking | B = Laya re-ranking of the same ten")


## 1. Environment

The notebook needs three things beyond a stock Colab: PyTorch Geometric (graph convolutions),
the Laya decision model, and pandas/matplotlib for reporting.

`laya` declares `torch>=2.0`, `transformers>=4.48` and `huggingface_hub>=0.20`. To make sure the
resolver can never trade Colab's CUDA torch build for a CPU one, the exact already-installed torch
version is written to a constraints file and reused for the install. A guard afterwards checks
that CUDA survived the install and offers a repair path if it did not.

In [ ]:
import importlib.util
import subprocess
import sys

import torch

TORCH_VERSION = torch.__version__
CUDA_BEFORE = torch.cuda.is_available()
print(f"pre-install  torch={TORCH_VERSION}  cuda={CUDA_BEFORE}")

# Freeze the CUDA build of torch that Colab already shipped.
constraints = WORK / "constraints.txt"
constraints.write_text(f"torch=={TORCH_VERSION}\n", encoding="utf-8")

cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "-c", str(constraints),
    "torch-geometric>=2.5",
    "pandas", "pyarrow", "matplotlib", "tqdm",
    "transformers>=4.48,<5",
    "laya",
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

# --- guard ------------------------------------------------------------------
# The running interpreter keeps the torch it already imported, so a wheel swap on
# disk is only visible from a *fresh* interpreter. Probe one, and repair if the
# CUDA build went away.
probe = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.cuda.is_available())"],
    capture_output=True, text=True, check=True,
)
fresh_version, fresh_cuda = probe.stdout.split()

if fresh_version != TORCH_VERSION or (CUDA_BEFORE and fresh_cuda != "True"):
    index = f"https://download.pytorch.org/whl/{TORCH_VERSION.split('+')[-1]}"
    print(f"torch on disk changed ({fresh_version}, cuda={fresh_cuda}); restoring {TORCH_VERSION} from {index}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
         f"torch=={TORCH_VERSION}", "--index-url", index],
        check=True,
    )
    raise RuntimeError("torch was repaired on disk — restart the runtime and re-run from cell 1.")

print(
    f"torch {TORCH_VERSION} (on disk {fresh_version}) | cuda {CUDA_BEFORE}"
    f" | torch-geometric {'ok' if importlib.util.find_spec('torch_geometric') else 'MISSING'}"
    f" | laya {'ok' if importlib.util.find_spec('laya') else 'MISSING'}"
)

## 2. Download the published weights and the test split

Two Hugging Face artefacts are pulled:

* **`jajostrains/Mathlib-Sexpr-GNN`** — the trained bundle. We take the
  `pointer-gat-gru/` directory (a `bundle.json` manifest, `config.json`, `model.safetensors`,
  `scorer.safetensors`, `summary.json`) plus the sibling `vocab/` directory, because
  `bundle.json` declares `"vocab_location": "shared"`.
* **`jajostrains/Mathlib-Normalized-Sexpr`** — the benchmark dataset; only
  `test/test-00000.parquet` (about 4.4 k proof states) is needed.

`bundle.json` carries a SHA-256 for the weights and for both vocabularies; the loader verifies
all three, so a partial or corrupt download fails loudly instead of silently degrading accuracy.

In [ ]:
from huggingface_hub import hf_hub_download

model_files = [
    f"{BUNDLE_NAME}/bundle.json",
    f"{BUNDLE_NAME}/config.json",
    f"{BUNDLE_NAME}/model.safetensors",
    f"{BUNDLE_NAME}/scorer.safetensors",
    f"{BUNDLE_NAME}/summary.json",
    "vocab/node_vocab.json",
    "vocab/tactic_vocab.json",
]
for rel in model_files:
    path = hf_hub_download(MODEL_REPO, rel, local_dir=MODEL_DIR)
    print(f"model   {rel:38s} -> {path}")

test_parquet = hf_hub_download(
    DATASET_REPO, f"{SPLIT}/{SPLIT}-00000.parquet",
    repo_type="dataset", local_dir=DATASET_DIR,
)
print(f"dataset {SPLIT}/{SPLIT}-00000.parquet -> {test_parquet}")

In [ ]:
df = pd.read_parquet(test_parquet)
assert len(df) > 0, "empty test split"

required = {"row_index", "theorem", "tactic", "text_state"}
missing = required - set(df.columns)
assert not missing, f"dataset is missing columns: {sorted(missing)}"

# Deterministic, *uniform* sample of the test split. The split is ordered by
# repository/theorem, so taking the first N rows would concentrate on a handful
# of files; a seeded random sample keeps the reported numbers comparable to the
# published full-split figure.
rng = np.random.default_rng(SEED)
if N_ROWS < len(df):
    positions = np.sort(rng.choice(len(df), size=N_ROWS, replace=False))
else:
    positions = np.arange(len(df))

eval_df = df.iloc[positions].reset_index(drop=True)
print(f"test split rows   : {len(df)}")
print(f"rows to evaluate  : {len(eval_df)}")
print(eval_df[["row_index", "theorem", "tactic"]].head(3).to_string(index=False))

## 3. The vendored core

The next cell writes `atp_core.py` next to this notebook. It is a trimmed copy of the pieces of
`maths_ai/gnn_inference/atp_lean_gnn` that offline tactic inference actually needs:

| vendored from | what |
|---|---|
| `state.py` | `parse_state` — split a proof state at the turnstile |
| `parser.py` | `ExprParser` — recursive-descent parser for Lean surface syntax |
| `graph.py` | `DAGBuilder`, `_classify_label`, `proof_state_to_dag` (text path) |
| `pyg.py` | `NODE_TYPE_TO_ID`, `dag_to_pyg` |
| `training.py` | `transform_edge_index` (`bidirectional` = forward + reversed edges) |
| `labels.py` | `normalize_tactic`, `encode_tactic_name` |
| `model.py` | `StateMeanAttentionReadout`, `GATv2StateClassifier` |
| `bundle.py` | `load_bundle` with SHA-256 verification, `build_model` |

What is deliberately **not** vendored: the Pantograph S-expression machinery (it needs a live
Lean REPL) and the argument/premise heads of the pointer model (they are trained, but this
notebook evaluates the *tactic family* ranking only).

In [ ]:
%%writefile atp_core.py
"""Self-contained core for the Colab notebook.

Vendored from ``maths_ai/gnn_inference/atp_lean_gnn`` (state.py, parser.py,
graph.py, pyg.py, training.py, labels.py, model.py) so the notebook has no
dependency on the repository.  Only the code paths actually needed for offline
tactic inference are kept:

* the *text* proof-state parser (``parse_state`` + ``ExprParser``),
* ``proof_state_to_dag`` in its text form,
* ``dag_to_pyg`` / ``transform_edge_index``,
* ``GATv2StateClassifier`` with the ``state_mean_attention`` readout,
* bundle loading with SHA-256 verification.

The S-expression branch of ``proof_state_to_dag`` is deliberately absent; see
the notebook section "Which graph construction the published bundle expects".
"""

from __future__ import annotations

import hashlib
import json
import re
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable

import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Batch, Data
from torch_geometric.nn import GATv2Conv, global_add_pool, global_max_pool, global_mean_pool
from torch_geometric.utils import softmax as graph_softmax


# ---------------------------------------------------------------------------
# state.py -- proof-state text parsing
# ---------------------------------------------------------------------------

TURNSTILES = ("\u22a2", "|-")


@dataclass(frozen=True)
class Hypothesis:
    name: str
    type_expr: str


@dataclass(frozen=True)
class ProofState:
    hypotheses: list[Hypothesis]
    goal: str


def _split_turnstile(state: str) -> tuple[str, str]:
    for turnstile in TURNSTILES:
        if turnstile in state:
            left, right = state.split(turnstile, maxsplit=1)
            return left.strip(), right.strip()
    return "", state.strip()


def parse_state(state: str) -> ProofState:
    hyp_block, goal = _split_turnstile(state)
    hypotheses: list[Hypothesis] = []

    if hyp_block:
        for raw_line in hyp_block.splitlines():
            line = raw_line.strip()
            if not line:
                continue

            if " : " in line:
                name, _, typ = line.partition(" : ")
                hypotheses.append(Hypothesis(name.strip(), typ.strip()))
                continue

            if ":" in line:
                name, _, typ = line.partition(":")
                hypotheses.append(Hypothesis(name.strip(), typ.strip()))
                continue

            hypotheses.append(Hypothesis(line, "Prop"))

    return ProofState(hypotheses=hypotheses, goal=goal)


# ---------------------------------------------------------------------------
# parser.py -- recursive-descent parser for Lean surface syntax
# ---------------------------------------------------------------------------

_TOKEN_RE = re.compile(
    r"""
    (?P<LPAREN>  \(                       ) |
    (?P<RPAREN>  \)                       ) |
    (?P<ARROW>   →|->                ) |
    (?P<COMMA>   ,                        ) |
    (?P<AT>      @                        ) |
    (?P<COLON>   :                        ) |
    (?P<IDENT>   [^\s()→@\[\]\u27e8\u27e9,;:]+ )
    """,
    re.VERBOSE,
)


def tokenize(expr: str) -> list[tuple[str, str]]:
    return [(match.lastgroup, match.group()) for match in _TOKEN_RE.finditer(expr)]


class ExprParser:
    """Recursive descent parser for a Lean-style expression.

    Grammar (simplified)::

        expr     := arrow
        arrow    := app ( ("->" | "→") app )*
        app      := atom+
        atom     := binder | IDENT | "(" expr ")" | "@" atom
        binder   := ("∀" | "∃" | "λ" | "let") atom ("," expr)?

    Binders such as ``∀ (q : Prop), body`` are parsed into
    ``App(∀, App(App(q, :), Prop), body)``.
    """

    _BINDER_LABELS = frozenset({"∀", "∃", "λ", "let"})

    def __init__(self, dag: "DAGBuilder"):
        self.dag = dag
        self.tokens: list[tuple[str, str]] = []
        self.pos = 0

    def parse(self, expr_str: str) -> int:
        self.tokens = tokenize(expr_str)
        self.pos = 0
        return self._parse_arrow()

    def _parse_arrow(self) -> int:
        left = self._parse_app()
        while self._peek_type() == "ARROW":
            self._consume()
            right = self._parse_app()
            left = self.dag.get_or_create("Arrow", (left, right))
        return left

    def _parse_app(self) -> int:
        func = self._parse_atom()
        if func is None:
            return self.dag.get_or_create("?", ())

        func_label = self.dag.nodes[func].label if func < len(self.dag.nodes) else ""
        if func_label in self._BINDER_LABELS and self._peek_type() == "LPAREN":
            var_decl = self._parse_atom()
            if var_decl is not None:
                func = self.dag.get_or_create("App", (func, var_decl))
                if self._peek_type() == "COMMA":
                    self._consume()
                    body = self._parse_arrow()
                    func = self.dag.get_or_create("App", (func, body))
                    return func

        while True:
            arg = self._parse_atom()
            if arg is None:
                break
            func = self.dag.get_or_create("App", (func, arg))
        return func

    def _parse_atom(self) -> int | None:
        token = self._peek()
        if token is None:
            return None

        token_type, token_value = token
        if token_type == "LPAREN":
            self._consume()
            node = self._parse_arrow()
            if self._peek_type() == "RPAREN":
                self._consume()
            return node

        if token_type == "AT":
            self._consume()
            inner = self._parse_atom()
            if inner is None:
                return self.dag.get_or_create("@", ())
            return self.dag.get_or_create("Explicit", (inner,))

        if token_type == "IDENT":
            self._consume()
            return self.dag.get_or_create(token_value, ())

        if token_type == "COLON":
            self._consume()
            return self.dag.get_or_create(":", ())

        return None

    def _peek(self) -> tuple[str, str] | None:
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def _peek_type(self) -> str | None:
        token = self._peek()
        return token[0] if token else None

    def _consume(self) -> tuple[str, str]:
        token = self.tokens[self.pos]
        self.pos += 1
        return token


# ---------------------------------------------------------------------------
# graph.py -- DAG construction
# ---------------------------------------------------------------------------

BINDER_KIND_UNKNOWN = -1
BINDER_KIND_NONE = 0
BINDER_KIND_FORALL = 1
BINDER_KIND_EXISTS = 2
BINDER_KIND_LAMBDA = 3
BINDER_KIND_LET = 4
BINDER_KIND_OTHER = 5


@dataclass(frozen=True)
class GraphNode:
    id: int
    label: str
    node_type: str
    children: tuple[int, ...] = field(default_factory=tuple)
    is_bound: int = BINDER_KIND_NONE
    binder_depth: int = 0
    binder_kind: int = BINDER_KIND_UNKNOWN


def _classify_label(label: str) -> str:
    if not label:
        return "var"
    if label in ("App", "Arrow", "Forall", "Explicit"):
        return "app"
    if label in ("Hyp", "Goal", "State"):
        return "meta"
    if label == "\u2115" or (label[0].isupper() and len(label) <= 2):
        return "type"
    if label[0].isupper():
        return "predicate"
    if label in ("+", "-", "*", "/", "=", "\u2264", "\u2265", "<", ">", "\u2227", "\u2228", "\u00ac"):
        return "operator"
    if label.startswith(":"):
        return "sbinder" if label in (":forall", ":lambda", ":let") else "sconst"
    return "var"


@dataclass
class DAGBuilder:
    """Hash-consed DAG.  Edges are ``(child_id, parent_id)`` pairs."""

    nodes: list[GraphNode] = field(default_factory=list)
    edges: list[tuple[int, int]] = field(default_factory=list)
    expression_root_id: int | None = None
    _memo: dict[tuple[str, str, tuple[int, ...]], int] = field(default_factory=dict)

    def add_node(
        self,
        label: str,
        children: tuple[int, ...],
        *,
        node_type: str | None = None,
        is_bound: int = BINDER_KIND_NONE,
        binder_depth: int = 0,
        binder_kind: int = BINDER_KIND_UNKNOWN,
        hash_cons: bool = True,
    ) -> int:
        resolved_node_type = node_type or _classify_label(label)
        key = (label, resolved_node_type, children)
        if hash_cons and key in self._memo:
            return self._memo[key]

        node_id = len(self.nodes)
        self.nodes.append(
            GraphNode(
                node_id,
                label,
                resolved_node_type,
                children,
                is_bound=is_bound,
                binder_depth=binder_depth,
                binder_kind=binder_kind,
            )
        )
        for child_id in children:
            self.edges.append((child_id, node_id))
        if hash_cons:
            self._memo[key] = node_id
        return node_id

    def get_or_create(self, label: str, children: tuple[int, ...], *, node_type: str | None = None) -> int:
        return self.add_node(label, children, node_type=node_type)

    @property
    def num_nodes(self) -> int:
        return len(self.nodes)

    @property
    def num_edges(self) -> int:
        return len(self.edges)

    def incoming_counts(self) -> Counter[int]:
        return Counter(parent_id for (_, parent_id) in self.edges)

    def outgoing_counts(self) -> Counter[int]:
        return Counter(child_id for (child_id, _) in self.edges)


def proof_state_to_dag(state: str) -> DAGBuilder:
    """Build a DAG from a pretty-printed proof state (text parser path)."""
    parsed = parse_state(state)

    dag = DAGBuilder()
    parser = ExprParser(dag)
    root_ids: list[int] = []

    for hypothesis in parsed.hypotheses:
        name_node = dag.get_or_create(hypothesis.name, ())
        type_node = (
            parser.parse(hypothesis.type_expr)
            if hypothesis.type_expr
            else dag.get_or_create("?", ())
        )
        hyp_node = dag.get_or_create("Hyp", (name_node, type_node))
        root_ids.append(hyp_node)

    goal_expr_node = parser.parse(parsed.goal)
    goal_node = dag.get_or_create("Goal", (goal_expr_node,))
    root_ids.append(goal_node)
    dag.get_or_create("State", tuple(root_ids))

    return dag


# ---------------------------------------------------------------------------
# pyg.py -- DAG -> PyG
# ---------------------------------------------------------------------------

NODE_TYPE_TO_ID = {
    "var": 0,
    "type": 1,
    "predicate": 2,
    "operator": 3,
    "app": 4,
    "meta": 5,
    "binder": 6,
    "const": 7,
    "sconst": 8,
    "sbinder": 9,
    "sapp": 10,
}


def build_vocab_from_labels(labels: Iterable[str], *, unk_token: str = "<UNK>") -> dict[str, int]:
    vocab = {unk_token: 0}
    for index, label in enumerate(sorted(set(labels)), start=1):
        vocab[label] = index
    return vocab


def dag_to_pyg(dag: DAGBuilder, vocab: dict[str, int]):
    """Convert a ``DAGBuilder`` to a ``torch_geometric.data.Data`` object."""
    x = torch.tensor([vocab.get(node.label, 0) for node in dag.nodes], dtype=torch.long)
    node_type = torch.tensor(
        [NODE_TYPE_TO_ID.get(node.node_type, NODE_TYPE_TO_ID["meta"]) for node in dag.nodes],
        dtype=torch.long,
    )
    is_bound = torch.tensor([node.is_bound for node in dag.nodes], dtype=torch.long)
    binder_depth = torch.clamp(
        torch.tensor([node.binder_depth for node in dag.nodes], dtype=torch.long), min=0
    )
    binder_kind = torch.clamp(
        torch.tensor([node.binder_kind for node in dag.nodes], dtype=torch.long), min=0
    )

    edge_pairs = list(dict.fromkeys(dag.edges))
    if edge_pairs:
        edge_index = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)

    return Data(
        x=x,
        edge_index=edge_index,
        node_type=node_type,
        is_bound=is_bound,
        binder_depth=binder_depth,
        binder_kind=binder_kind,
        num_nodes=dag.num_nodes,
    )


def root_state_node_index(dag: DAGBuilder) -> int:
    """Index of the single ``State`` node that is not the child of any edge."""
    state_ids = [node.id for node in dag.nodes if node.label == "State"]
    source_ids = {source for source, _ in dag.edges}
    roots = [node_id for node_id in state_ids if node_id not in source_ids]
    if len(roots) != 1:
        raise ValueError(f"Expected exactly one root State node, found {len(roots)}.")
    return roots[0]


# ---------------------------------------------------------------------------
# training.py -- edge-mode transform
# ---------------------------------------------------------------------------

def transform_edge_index(edge_index: torch.Tensor, *, edge_mode: str) -> torch.Tensor:
    if edge_mode == "forward":
        return edge_index.to(dtype=torch.long).contiguous()
    if edge_mode != "bidirectional":
        raise ValueError(f"Unsupported edge mode '{edge_mode}'.")
    if edge_index.numel() == 0:
        return edge_index.to(dtype=torch.long).contiguous()

    forward = edge_index.to(dtype=torch.long)
    reverse = forward[[1, 0], :]
    return torch.cat([forward, reverse], dim=1).contiguous()


# ---------------------------------------------------------------------------
# labels.py -- tactic normalization
# ---------------------------------------------------------------------------

EMPTY_TACTIC = "<EMPTY_TACTIC>"
UNKNOWN_TACTIC = "<UNK_TACTIC>"
TACTIC_TOKEN_RE = re.compile(r"[A-Za-z0-9_.'!?]+")


def normalize_tactic(raw: str) -> str:
    text = raw.strip()
    if not text:
        return EMPTY_TACTIC
    match = TACTIC_TOKEN_RE.search(text)
    if match is None:
        return EMPTY_TACTIC
    return match.group(0)


def encode_tactic_name(tactic_name: str, tactic_vocab: dict[str, int]) -> int:
    return tactic_vocab.get(tactic_name, tactic_vocab[UNKNOWN_TACTIC])


# ---------------------------------------------------------------------------
# model.py -- GATv2 state classifier + attention readout
# ---------------------------------------------------------------------------

VALID_READOUTS = (
    "state",
    "state_mean_attention",
    "state_max_attention",
    "state_mean_max_attention",
)


def _embed_binder_depth(embedding: nn.Embedding, binder_depth: torch.Tensor) -> torch.Tensor:
    bounded_depth = binder_depth.clamp(min=0, max=embedding.num_embeddings - 1)
    return embedding(bounded_depth)


def _state_node_embeddings(node_embeddings: torch.Tensor, data) -> torch.Tensor:
    if not hasattr(data, "state_node_index"):
        raise ValueError("Batched graph data is missing the 'state_node_index' attribute.")
    state_node_index = data.state_node_index
    if not torch.is_tensor(state_node_index):
        state_node_index = torch.tensor(
            [int(state_node_index)], device=node_embeddings.device, dtype=torch.long
        )
    else:
        state_node_index = state_node_index.to(
            device=node_embeddings.device, dtype=torch.long
        ).view(-1)
    return node_embeddings.index_select(0, state_node_index)


class StateMeanAttentionReadout(nn.Module):
    """Fuse state and attention summaries with optional mean/max pooling."""

    def __init__(self, hidden_dim: int, *, include_mean: bool = True, include_max: bool = False) -> None:
        super().__init__()
        self.include_mean = include_mean
        self.include_max = include_max
        self.node_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.state_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.attention_score = nn.Linear(hidden_dim, 1, bias=False)
        summary_count = 2 + int(include_mean) + int(include_max)
        self.fusion = nn.Linear(hidden_dim * summary_count, hidden_dim)
        self.normalization = nn.LayerNorm(hidden_dim)

    def forward(
        self,
        node_embeddings: torch.Tensor,
        state_embeddings: torch.Tensor,
        batch_index: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch_index = batch_index.to(device=node_embeddings.device, dtype=torch.long)
        graph_count = state_embeddings.size(0)
        state_per_node = state_embeddings.index_select(0, batch_index)
        scores = self.attention_score(
            torch.tanh(
                self.node_projection(node_embeddings) + self.state_projection(state_per_node)
            )
        ).squeeze(-1)
        attention_weights = graph_softmax(scores, batch_index, num_nodes=graph_count)
        weighted_embeddings = global_add_pool(
            attention_weights.unsqueeze(-1) * node_embeddings, batch_index, size=graph_count
        )
        summaries = [state_embeddings]
        if self.include_mean:
            summaries.append(global_mean_pool(node_embeddings, batch_index, size=graph_count))
        if self.include_max:
            summaries.append(global_max_pool(node_embeddings, batch_index, size=graph_count))
        summaries.append(weighted_embeddings)
        combined = torch.cat(summaries, dim=-1)
        fused = self.fusion(combined)
        return F.gelu(self.normalization(fused)), attention_weights


class GATv2StateClassifier(nn.Module):
    def __init__(
        self,
        *,
        num_node_labels: int,
        num_tactics: int,
        num_node_types: int = len(NODE_TYPE_TO_ID),
        num_binder_kinds: int = 6,
        max_binder_depth: int = 10,
        hidden_dim: int = 256,
        num_layers: int = 4,
        dropout: float = 0.2,
        heads: int = 8,
        use_node_type: bool = True,
        readout: str = "state",
    ) -> None:
        super().__init__()
        if num_layers < 1:
            raise ValueError("GATv2StateClassifier requires at least one message-passing layer.")
        if heads < 1:
            raise ValueError("GATv2StateClassifier requires at least one attention head.")
        if hidden_dim % heads != 0:
            raise ValueError(
                "GATv2StateClassifier requires hidden_dim to be divisible by heads "
                f"(got hidden_dim={hidden_dim}, heads={heads})."
            )
        normalized_readout = readout.lower().strip()
        if normalized_readout not in VALID_READOUTS:
            raise ValueError(
                f"GATv2StateClassifier readout must be one of: {', '.join(VALID_READOUTS)}."
            )

        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.readout_mode = normalized_readout
        self.label_embedding = nn.Embedding(num_node_labels, hidden_dim)
        self.node_type_embedding = (
            nn.Embedding(num_node_types, hidden_dim) if use_node_type else None
        )

        self.is_bound_embedding = nn.Embedding(2, hidden_dim)
        self.binder_depth_embedding = nn.Embedding(max_binder_depth, hidden_dim)
        self.binder_kind_embedding = nn.Embedding(num_binder_kinds, hidden_dim)

        self.convs = nn.ModuleList(
            GATv2Conv(
                hidden_dim, self.head_dim, heads=heads, dropout=dropout, concat=True
            )
            for _ in range(num_layers)
        )
        self.dropout = nn.Dropout(dropout)
        if self.readout_mode == "state":
            self.global_readout = None
        else:
            self.global_readout = StateMeanAttentionReadout(
                hidden_dim,
                include_mean="mean" in self.readout_mode,
                include_max="max" in self.readout_mode,
            )
        self.classifier = nn.Linear(hidden_dim, num_tactics)

    def encode_nodes(self, data) -> torch.Tensor:
        x = self.label_embedding(data.x)
        if self.node_type_embedding is not None:
            x = x + self.node_type_embedding(data.node_type)

        if hasattr(data, "is_bound"):
            x = x + self.is_bound_embedding(data.is_bound)
        if hasattr(data, "binder_depth"):
            x = x + _embed_binder_depth(self.binder_depth_embedding, data.binder_depth)
        if hasattr(data, "binder_kind"):
            x = x + self.binder_kind_embedding(data.binder_kind)

        for conv in self.convs:
            x = conv(x, data.edge_index)
            x = F.relu(x)
            x = self.dropout(x)
        return x

    def readout_with_details(self, node_embeddings: torch.Tensor, data):
        state_embeddings = _state_node_embeddings(node_embeddings, data)
        if self.global_readout is None:
            return state_embeddings, {}
        batch_index = getattr(data, "batch", None)
        if batch_index is None:
            batch_index = torch.zeros(
                node_embeddings.size(0), device=node_embeddings.device, dtype=torch.long
            )
        graph_embeddings, attention_weights = self.global_readout(
            node_embeddings, state_embeddings, batch_index
        )
        return graph_embeddings, {"attention_weights": attention_weights}

    def readout(self, node_embeddings: torch.Tensor, data) -> torch.Tensor:
        graph_embeddings, _ = self.readout_with_details(node_embeddings, data)
        return graph_embeddings

    def forward_with_readout_details(self, data):
        node_embeddings = self.encode_nodes(data)
        graph_embeddings, details = self.readout_with_details(node_embeddings, data)
        return self.classifier(self.dropout(graph_embeddings)), details

    def forward(self, data) -> torch.Tensor:
        logits, _ = self.forward_with_readout_details(data)
        return logits


# ---------------------------------------------------------------------------
# bundle.py -- loading the published Hugging Face bundle
# ---------------------------------------------------------------------------

def stable_vocab_sha256(vocab: dict) -> str:
    """Hash a vocabulary exactly the way the training run hashed it."""
    payload = json.dumps(vocab, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


@dataclass
class Bundle:
    directory: Path
    manifest: dict
    config: dict
    node_vocab: dict[str, int]
    tactic_vocab: dict[str, int]
    id_to_tactic: dict[int, str]

    @property
    def model_dir(self) -> Path:
        return self.directory


def resolve_vocab_paths(bundle_dir: Path, manifest: dict) -> tuple[Path, Path]:
    """Mirror the repository's ``resolve_vocab_paths``.

    ``bundle.json`` records ``vocab_location``; ``"shared"`` means the vocab
    lives next to the bundle directory rather than inside it.
    """
    location = manifest.get("vocab_location", "local")
    if location == "shared":
        candidate = bundle_dir.parent / "vocab"
        if (candidate / "node_vocab.json").is_file():
            return candidate / "node_vocab.json", candidate / "tactic_vocab.json"
    if (bundle_dir / "node_vocab.json").is_file():
        return bundle_dir / "node_vocab.json", bundle_dir / "tactic_vocab.json"
    raise FileNotFoundError(f"Could not locate vocab files for {bundle_dir}")


def load_bundle(bundle_dir: str | Path, *, verify_hashes: bool = True) -> Bundle:
    bundle_dir = Path(bundle_dir)
    manifest = json.loads((bundle_dir / "bundle.json").read_text(encoding="utf-8"))
    config = json.loads((bundle_dir / manifest["config"]).read_text(encoding="utf-8"))

    node_path, tactic_path = resolve_vocab_paths(bundle_dir, manifest)
    node_vocab = json.loads(node_path.read_text(encoding="utf-8"))
    tactic_vocab = json.loads(tactic_path.read_text(encoding="utf-8"))

    if verify_hashes:
        weights = bundle_dir / manifest["weights"]
        if sha256_file(weights) != manifest["weights_sha256"]:
            raise ValueError(f"SHA-256 mismatch for {weights}")
        if stable_vocab_sha256(node_vocab) != manifest["node_vocab_sha256"]:
            raise ValueError(f"SHA-256 mismatch for {node_path}")
        if stable_vocab_sha256(tactic_vocab) != manifest["tactic_vocab_sha256"]:
            raise ValueError(f"SHA-256 mismatch for {tactic_path}")
        if len(node_vocab) != manifest["num_node_labels"]:
            raise ValueError("node_vocab size disagrees with bundle manifest")
        if len(tactic_vocab) != manifest["num_tactics"]:
            raise ValueError("tactic_vocab size disagrees with bundle manifest")

    return Bundle(
        directory=bundle_dir,
        manifest=manifest,
        config=config,
        node_vocab=node_vocab,
        tactic_vocab=tactic_vocab,
        id_to_tactic={int(v): k for k, v in tactic_vocab.items()},
    )


def build_model(bundle: Bundle, *, strict: bool = True) -> GATv2StateClassifier:
    """Rebuild the classifier and load only the ``backbone.*`` weights."""
    from safetensors.torch import load_file

    model_cfg = bundle.config["model"]
    model = GATv2StateClassifier(
        num_node_labels=len(bundle.node_vocab),
        num_tactics=len(bundle.tactic_vocab),
        num_node_types=len(NODE_TYPE_TO_ID),
        hidden_dim=model_cfg["hidden_dim"],
        num_layers=model_cfg["num_layers"],
        dropout=model_cfg["dropout"],
        use_node_type=bundle.config["use_node_type"],
        heads=model_cfg["heads"],
        readout=model_cfg["readout"],
    )

    raw = load_file(str(bundle.directory / bundle.manifest["weights"]))
    backbone = {
        key[len("backbone."):]: value
        for key, value in raw.items()
        if key.startswith("backbone.")
    }
    dropped = sorted({key.split(".", 1)[0] for key in raw if not key.startswith("backbone.")})
    result = model.load_state_dict(backbone, strict=strict)
    if strict and (result.missing_keys or result.unexpected_keys):
        raise ValueError(f"Weight mismatch: missing={result.missing_keys} unexpected={result.unexpected_keys}")
    model.eval()
    return model, dropped


# ---------------------------------------------------------------------------
# Inference helpers
# ---------------------------------------------------------------------------

def prepare_data(dag: DAGBuilder, node_vocab: dict[str, int], *, edge_mode: str = "bidirectional"):
    """``dag`` -> PyG ``Data`` exactly the way the training pipeline did."""
    data = dag_to_pyg(dag, node_vocab)
    data.state_node_index = torch.tensor([root_state_node_index(dag)], dtype=torch.long)
    data.edge_index = transform_edge_index(data.edge_index, edge_mode=edge_mode)
    return data


@torch.no_grad()
def predict_probs(model: GATv2StateClassifier, data_list: list[Data], *, batch_size: int = 64,
                  device: torch.device | str = "cpu") -> torch.Tensor:
    model.eval()
    model.to(device)
    outputs: list[torch.Tensor] = []
    for start in range(0, len(data_list), batch_size):
        batch = Batch.from_data_list(data_list[start:start + batch_size]).to(device)
        logits = model(batch)
        outputs.append(torch.softmax(logits, dim=-1).cpu())
    return torch.cat(outputs, dim=0) if outputs else torch.empty((0, 0))


def topk_predictions(probs: torch.Tensor, bundle: Bundle, k: int) -> tuple[list[list[str]], list[list[float]]]:
    k = min(k, probs.shape[-1])
    values, indices = torch.topk(probs, k, dim=-1)
    names = [[bundle.id_to_tactic[int(i)] for i in row] for row in indices]
    scores = [[float(v) for v in row] for row in values]
    return names, scores

In [ ]:
import importlib
import sys

sys.path.insert(0, str(WORK))
import atp_core as core

if hasattr(core, "__file__"):          # re-run safe
    core = importlib.reload(core)

# 1. Manifest + SHA-256 verification of weights and both vocabularies.
bundle = core.load_bundle(BUNDLE_DIR, verify_hashes=True)
print(f"bundle      : {bundle.manifest['model_type']}  ({BUNDLE_DIR})")
print(f"run / epoch : {bundle.manifest['source_run']} / {bundle.manifest['source_epoch']}")
print(f"vocab       : {len(bundle.node_vocab)} node labels, {len(bundle.tactic_vocab)} tactics")
print(f"edge_mode   : {bundle.config['edge_mode']}   readout: {bundle.config['model']['readout']}")
print(f"published val top-1: {bundle.manifest['val_metrics']['tactic_top1_accuracy']:.4f}")
print("SHA-256 of model.safetensors + node_vocab + tactic_vocab : OK")

# 2. Rebuild the classifier and load exactly the backbone weights.
model, dropped = core.build_model(bundle, strict=True)
n_params = sum(p.numel() for p in model.parameters())
print(f"dropped non-backbone prefixes : {dropped}")
print(f"loaded parameters             : {n_params:,}")

# 3. End-to-end smoke test on a hand-written proof state.
smoke_state = "α : Type\ninst✝ : Group α\nh : α = α\n⊢ α = α"
dag = core.proof_state_to_dag(smoke_state)
data = core.prepare_data(dag, bundle.node_vocab, edge_mode=bundle.config["edge_mode"])

assert data.num_nodes == dag.num_nodes
assert data.edge_index.numel() == 0 or int(data.edge_index.max()) < data.num_nodes
assert data.state_node_index.numel() == 1
assert bundle.node_vocab["State"] in data.x.tolist()
# bidirectional: forward edges are kept and reversed copies appended
assert dag.num_edges <= data.edge_index.size(1) <= 2 * dag.num_edges, (
    data.edge_index.size(1), dag.num_edges
)
assert int(data.x.min()) >= 0 and int(data.x.max()) < len(bundle.node_vocab)

probs = core.predict_probs(model, [data], batch_size=1, device=DEVICE)
assert probs.shape == (1, len(bundle.tactic_vocab))
assert torch.allclose(probs.sum(dim=-1), torch.ones(1), atol=1e-5)

names, scores = core.topk_predictions(probs, bundle, 5)
print("smoke top-5 :", ", ".join(f"{n} ({s:.3f})" for n, s in zip(names[0], scores[0])))
print("smoke test  : PASSED")

### Verified finding: which graph construction the published bundle expects

The Hugging Face model card describes the graphs as *S-expression goal-state DAGs*, and the
dataset ships `model_goal_sexp` / `model_hyp_sexps` columns for exactly that. Feeding them to
`proof_state_to_dag(..., goal_sexp=..., hyp_sexps=...)` does **not** reproduce the published
accuracy (measured on a 2 200-row systematic sample of the test split):

| graph construction | nodes mapped to `<UNK>` | top-1 | top-5 | top-10 |
|---|---|---|---|---|
| S-expression path (`model_goal_sexp`, `model_hyp_sexps`) | **74.0 %** | 0.156 | 0.376 | 0.554 |
| **text path (`text_state`)** — used below | **1.4 %** | **0.295** | **0.686** | **0.839** |
| published figure (`summary.json`, full upstream split, 4 506 rows) | – | 0.315 | 0.711 | – |

The 0.020 gap between 0.295 and 0.315 is ~2 standard errors and is explained by the sample
(2 200 of 4 392 rows, in contiguous blocks) plus the fact that the published number was measured
on the upstream 4 506-row split. A separate 100-row slice scored 0.310 / 0.730 / 0.900.

The cause is visible without running anything: the published `node_vocab.json` simply has no
S-expression vocabulary. The cell below prints the evidence — the labels the S-expression path
would emit are all absent, while the labels the *text* parser emits are all present, which means
the `best.pt` behind `model.safetensors` was trained on text-parser graphs.

Because of this the notebook builds graphs from `text_state`, the pretty-printed proof state,
which is also the only input that needs no Lean/Pantograph runtime.

In [ ]:
SEXP_LABELS = [
    ":app", ":arg", ":forall", ":lambda", ":c", ":fv", ":sort", ":lit",
    "FV0", "FV1", "HypRole:explicit", "HypRole:instance",
    "ArgRole::explicit", "ArgPosition:0", "BinderRole::explicit",
]
TEXT_LABELS = [
    "App", "Arrow", "Forall", "Explicit", "State", "Goal", "Hyp",
    "∀", ":", "?",
]

vocab = bundle.node_vocab
sexp_hits = {label: label in vocab for label in SEXP_LABELS}
text_hits = {label: label in vocab for label in TEXT_LABELS}

print("S-expression path labels present in node_vocab :",
      sum(sexp_hits.values()), "/", len(sexp_hits))
for label, present in sexp_hits.items():
    print(f"    {label:<24s} {'FOUND' if present else 'absent'}")

print("\nText-parser path labels present in node_vocab  :",
      sum(text_hits.values()), "/", len(text_hits))
for label, present in text_hits.items():
    print(f"    {label:<24s} {'FOUND' if present else 'absent'}")

assert sum(sexp_hits.values()) == 0, "unexpected: S-expression labels are in the vocab"
assert all(text_hits.values()), "unexpected: a text-parser label is missing"
print("\n=> the published bundle was trained on TEXT-parser graphs.")

## 4. Test rows → PyG graphs

For every sampled row:

1. `normalize_tactic(row["tactic"])` gives the tactic *family* (`"rw [foo, bar]" → "rw"`),
   mapped through `tactic_vocab` to the target id (`0` = `<UNK_TACTIC>`, excluded from metrics —
   this mirrors the training/evaluation convention).
2. `proof_state_to_dag(row["text_state"])` parses the pretty-printed state at the turnstile and
   builds a hash-consed DAG: one `Hyp` node per local, one `Goal` node, one root `State` node
   whose children are all of them.
3. `prepare_data` converts it to a PyG `Data` object — label ids in `x`, a coarse node type in
   `node_type`, binder features, the DAG's `(child, parent)` edges, and then
   `transform_edge_index(..., "bidirectional")` appends the reversed copies because the config
   says `"edge_mode": "bidirectional"`.
4. `state_node_index` points at the root `State` node; the readout pools from there.

The graphs are cached so re-running the later cells does not re-parse anything.

In [ ]:
from tqdm.auto import tqdm

GRAPH_CACHE = CACHE_DIR / f"graphs_{SPLIT}_{N_ROWS}_seed{SEED}.pt"
failures: list[tuple[int, str]] = []

if GRAPH_CACHE.exists():
    payload = torch.load(GRAPH_CACHE, map_location="cpu", weights_only=False)
    assert payload["positions"] == positions.tolist(), "cached sample differs from this run's sample"
    graphs = payload["graphs"]
    gt_ids = payload["gt_ids"]
    gt_names = payload["gt_names"]
    state_texts = payload["state_texts"]
    print(f"loaded {len(graphs)} cached graphs from {GRAPH_CACHE}")
else:
    graphs, gt_ids, gt_names, state_texts = [], [], [], []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="build graphs"):
        tactic_name = core.normalize_tactic(row["tactic"])
        target = core.encode_tactic_name(tactic_name, bundle.tactic_vocab)
        try:
            dag = core.proof_state_to_dag(row["text_state"])
            data = core.prepare_data(dag, bundle.node_vocab, edge_mode=bundle.config["edge_mode"])
        except Exception as exc:                                # pragma: no cover
            failures.append((int(row["row_index"]), f"{type(exc).__name__}: {exc}"))
            continue

        data.y = torch.tensor([target], dtype=torch.long)
        data.row_index = int(row["row_index"])
        graphs.append(data)
        gt_ids.append(target)
        gt_names.append(tactic_name)
        state_texts.append(str(row["text_state"]))

    torch.save(
        {"positions": positions.tolist(), "graphs": graphs, "gt_ids": gt_ids,
         "gt_names": gt_names, "state_texts": state_texts},
        GRAPH_CACHE,
    )
    print(f"cached {len(graphs)} graphs -> {GRAPH_CACHE}")

assert len(graphs) > 0, "no graphs were built"
assert len(graphs) == len(gt_ids) == len(gt_names) == len(state_texts)

node_counts = [int(d.num_nodes) for d in graphs]
edge_counts = [int(d.edge_index.size(1)) for d in graphs]
unk_rate = float((torch.cat([d.x for d in graphs]) == 0).float().mean())
n_known = sum(1 for g in gt_ids if g != 0)

print(f"graphs            : {len(graphs)}  ({n_known} with a known target)")
print(f"nodes per graph   : mean {np.mean(node_counts):.1f}  max {max(node_counts)}")
print(f"edges per graph   : mean {np.mean(edge_counts):.1f}  max {max(edge_counts)}")
print(f"<UNK> label rate  : {unk_rate:.4f}   (text path; ~0.74 for the S-expression path)")
if failures:
    print(f"skipped rows      : {len(failures)} -> {failures[:5]}")

## 5. Stage 1 — the GNN's top-10 pool

One forward pass per batch returns a softmax over all **228** tactic families; the ten
highest-scoring families become the candidate pool. Both systems below work only on these
ten, and neither ever sees tactic number eleven or lower.

Three things defined in the next cell are reused throughout the notebook:

* `known_mask` — rows whose target is in the tactic vocabulary (`<UNK_TACTIC>` targets are
  excluded, exactly as in training/evaluation);
* `candidate_mask` — rows whose target actually made it into the pool. This is the shared
  **pool ceiling**: no system can advance a target it never received, so both systems report
  the same `recall@10`;
* `evaluate(rankings, mask)` — the metrics helper used for both systems:
  * `recall@k` — share of counted rows whose target appears in the system's ranking up to `k`,
  * `mrr@10` — mean reciprocal rank of the target within the ten,
  * `n` — number of rows counted.


In [ ]:
model.to(DEVICE)
probs = core.predict_probs(model, graphs, batch_size=BATCH_SIZE, device=DEVICE)
assert probs.shape == (len(graphs), len(bundle.tactic_vocab))

# Stage 1: cut the pool. The full ranking over all 228 tactics is used only to pick
# these ten; everything downstream (both systems) sees nothing outside the pool.
gnn_rankings, gnn_scores = core.topk_predictions(probs, bundle, POOL_K)
known_mask = [g != 0 for g in gt_ids]
candidate_mask = [gt_names[i] in gnn_rankings[i] for i in range(len(gt_names))]


def rank_of(target: str, ranking: list[str]) -> int | None:
    try:
        return ranking.index(target) + 1
    except ValueError:
        return None


def evaluate(rankings, mask, ks=TOP_K, mrr_k=TOP_K[0]):
    """Metrics for one system. ``mask`` selects the rows that count."""
    idx = [i for i, keep in enumerate(mask) if keep]
    result = {"n": len(idx)}
    if not idx:
        return result
    for k in ks:
        result[f"recall@{k}"] = float(
            np.mean([gt_names[i] in rankings[i][:k] for i in idx])
        )
    rr = []
    for i in idx:
        r = rank_of(gt_names[i], rankings[i][:mrr_k])
        rr.append(0.0 if r is None else 1.0 / r)
    result[f"mrr@{mrr_k}"] = float(np.mean(rr))
    return result


n_known = sum(known_mask)
n_in_pool = sum(1 for i in range(len(gt_names)) if known_mask[i] and candidate_mask[i])
print(f"rows                : {len(gt_names)}")
print(f"known targets       : {n_known}")
print(f"target in top-{POOL_K}      : {n_in_pool}  "
      f"({100 * n_in_pool / max(n_known, 1):.1f}% of known "
      f"— the shared recall@{POOL_K} ceiling)")

show = 5
for i in range(show):
    if gt_names[i] in gnn_rankings[i][:1]:
        flag = "top-1"
    elif gt_names[i] in gnn_rankings[i]:
        flag = f"pool #{gnn_rankings[i].index(gt_names[i]) + 1}"
    else:
        flag = "outside pool"
    print(f"\nrow {graphs[i].row_index:>6}  gt={gt_names[i]:<18} [{flag}]  "
          f"pool: {', '.join(gnn_rankings[i])}")


## 6. System B — Laya re-ranks the ten

[Laya](https://github.com/NandhaKishorM/laya) is a non-autoregressive decision model: you hand it
a *state* and a set of typed questions, it answers every question in a single forward pass and
returns calibrated probabilities. System B uses its `choice` question type on the pool:

```python
{"type": "choice",
 "instructions": "Which Lean 4 tactic family best matches this proof step?",
 "criteria": {"simp": "simplify with a lemma set",
              "rw":   "rewrite with an equation",
              ...}}
```

Three properties matter for this experiment:

* **The answer carries a full distribution.** `answer["probabilities"]` maps every candidate to
  its (temperature-calibrated) probability, so Laya produces a *ranking* of the ten, not just a
  label — that ranking is System B's ordering of the pool: its first five are what System B
  advances, and its first entry is System B's final pick.
* **States are truncated from the right** (`state_ids[:room]`), so the goal is written first and
  the local context after it — a long context can be cut without ever losing the goal.
* **One call per row.** Each row has its *own* option set, so the calls cannot be batched
  (`predict_batch` requires one shared question set). 500 rows ≈ 500 forward passes, about a
  minute including warm-up.

### Why Laya only ever sees the ten

A `choice` question spends **one shared `head_max_len` budget** (192 tokens on the English
checkpoint) on *all* of its options, and `Agent._encode_state` refuses the whole question when
they no longer fit:

```
question 'tactic' options exceed head_max_len=192
```

With a one-line gloss that ceiling lands at roughly **20 options**. The published tactic
vocabulary has **226 usable labels**, so "Laya picks from every tactic" cannot be asked as one
question — it would need a ~1,600-token option budget on a checkpoint trained at `max_len=512`,
and the tokens it eats would come straight out of the proof state.

Laya does ship `predict_shortlist` for exactly this case (embed the state and every option with
`embed_fn_from_agent`, keep the top `k`, then run `predict` on the reduced set). But the
shortlist is chosen by an **encoder cosine similarity** *before* Laya sees anything, so the
result would be "a bi-encoder shortlists, then Laya reorders 20" — not Laya selecting from all
tactics. That benchmark is therefore **dropped**, and Laya is evaluated where it is sound: as a
re-ranker of the ten candidates the GNN actually proposed.

Checkpointing: rankings are written to parquet every 50 rows, so an interrupted run resumes
(`cache/laya_*_k10.parquet`).


In [ ]:
import laya

agent = laya.load(LAYA_MODEL, device=LAYA_DEVICE)
cfg = agent.cfg
print(f"checkpoint : {LAYA_MODEL}")
print(f"device     : {LAYA_DEVICE}")
print(f"max_len    : {cfg.get('max_len')}   head_max_len : {cfg.get('head_max_len')}")
TACTIC_GLOSS = {
    ".": "move on to the next goal in the list",
    "C_simp": "simplify commutativity goals",
    "abel": "normalize an abelian group expression",
    "abel_nf": "abelian normal form, structural rewriting",
    "ac_rfl": "reflexivity up to associativity and commutativity",
    "aesop": "automated search of standard proof steps",
    "aesop_cat": "aesop with a category-theory rule set",
    "aesop_mat": "aesop with a matroid rule set",
    "all_goals": "run the tactic on every open goal",
    "any_goals": "run the tactic on the goals where it succeeds",
    "apply": "apply a lemma or hypothesis to the goal",
    "apply_assumption": "apply a hypothesis that solves the goal",
    "apply_fun": "apply a function to both sides of an equation",
    "apply_mod_cast": "apply a lemma after normalizing casts",
    "apply_rules": "apply one of a list of lemmas",
    "assumption": "close the goal with a matching hypothesis",
    "assumption'": "assumption that may leave extra goals",
    "assumption_mod_cast": "assumption after normalizing casts",
    "beta_reduce": "beta-reduce lambdas in the goal",
    "bitwise_assoc_tac": "reassociate bitwise operations",
    "borelize": "rewrite measures into Borel sets",
    "by_cases": "split on a decidable proposition",
    "by_contra": "prove the goal by contradiction",
    "by_contra!": "prove by contradiction using classical logic",
    "calc": "start a calculation chain",
    "cancel_denoms": "cancel common denominators",
    "case": "work on one named case of the goal",
    "case'": "work on a case, keeping the others",
    "cases": "destruct a value into its constructors",
    "cases'": "destruct, keeping the other goals",
    "cases_type": "case split on the goal's type",
    "casesm": "case split on several hypotheses",
    "change": "replace the goal with a definitionally equal one",
    "choose": "extract witnesses from a hypothesis into variables",
    "choose!": "choose witnesses and register instances",
    "classical": "enable classical reasoning in the goal",
    "clear": "remove an unused hypothesis",
    "clear!": "clear a hypothesis, ignoring dependencies",
    "clear_value": "erase local definitions, keeping their types",
    "coherence": "close a coherence-condition goal",
    "compute_degree": "compute the degree of a polynomial",
    "compute_degree!": "compute a degree with normalization",
    "congr": "split an equality by congruence of subterms",
    "congr!": "congruence combined with rewriting",
    "congrm": "congruence guided by a matching schema",
    "constructor": "apply the goal's first constructor",
    "continuity": "prove continuity of an expression",
    "contradiction": "close the goal from contradictory hypotheses",
    "contrapose": "replace the goal with its contrapositive",
    "contrapose!": "contrapose using classical logic",
    "conv": "enter conversion (targeted rewriting) mode",
    "conv_lhs": "convert the left-hand side of the goal",
    "conv_rhs": "convert the right-hand side of the goal",
    "convert": "prove a goal like a hint, allowing extra arguments",
    "convert_to": "reduce the goal to an equivalent statement",
    "decide": "close the goal by evaluation",
    "delta": "unfold a constant definitionally",
    "derivative_simp": "simplify derivatives",
    "done": "succeed only if no goals remain",
    "dsimp": "definitional simplification of the goal",
    "erw": "rewrite with explicit reducibility",
    "eta_expand": "eta-expand terms in the goal",
    "eval_simp": "simplify by evaluation",
    "exact": "close the goal with an exact term",
    "exact_mod_cast": "close the goal after normalizing casts",
    "exacts": "close the goal with one of several terms",
    "exfalso": "reduce the goal to False",
    "exists": "prove an existential with an explicit witness",
    "existsi": "provide a witness for an existential goal",
    "ext": "extensionality: prove equality part by part",
    "ext1": "one step of extensionality",
    "fapply": "apply with higher-order unification",
    "fconstructor": "constructor with higher-order unification",
    "field_simp": "normalize expressions in a field",
    "filter_upwards": "narrow a filter statement using hypotheses",
    "fin_cases": "case split over a finite type",
    "first": "try tactics in order and keep the first success",
    "focus": "run a tactic on the first goal only",
    "fun_prop": "prove properties of a function",
    "funext": "prove two functions equal by equal values",
    "gcongr": "generalized congruence on inequalities",
    "generalize": "replace a subterm by a fresh variable",
    "generalize_proofs": "turn proof arguments into hypotheses",
    "ghost_calc": "ghost computation inside a calculation",
    "ghost_fun_tac": "ghost-function rewriting",
    "ghost_simp": "ghost simplification",
    "group": "normalize into group normal form",
    "have": "introduce an intermediate claim",
    "have'": "introduce a claim, keeping the main goal first",
    "haveI": "introduce a claim and register it as an instance",
    "if": "split an if-then-else",
    "induction": "prove by induction on a variable",
    "induction'": "induction with reordered step goals",
    "infer_instance": "fill the goal with a typeclass instance",
    "infer_param": "infer a parameter of a structure",
    "inhabit": "produce an inhabitant of a nonempty type",
    "init_ring": "initialize the ring normalizer",
    "injection": "derive equalities from constructor injectivity",
    "injections": "apply injectivity to all constructors",
    "interval_cases": "case split over an interval of naturals",
    "intro": "introduce a binder or hypothesis",
    "intros": "introduce all binders",
    "introv": "introduce hypotheses with chosen names",
    "isBoundedDefault": "default boundedness check",
    "iterate": "apply a tactic repeatedly",
    "left": "prove a disjunction by its left disjunct",
    "let": "introduce a local definition",
    "letI": "introduce a local definition as an instance",
    "lift": "lift a value along an order embedding",
    "lift_lets": "inline local let bindings",
    "linarith": "linear arithmetic reasoning",
    "linear_combination": "prove equality from a linear combination",
    "map_fun_tac": "map a function through a structure",
    "map_simp": "simplify mapped expressions",
    "match": "pattern match on a value",
    "matrix_simp": "simplify matrix expressions",
    "measurability": "prove measurability",
    "mem_tac": "membership reasoning",
    "mfld_set_tac": "manifold set reasoning",
    "mono": "monotonicity step",
    "move_mul": "reorder multiplications",
    "mv_bisim": "mutual-variant bisimulation",
    "next": "select the next goal",
    "nlinarith": "nonlinear arithmetic reasoning",
    "nofun": "no-function fallback step",
    "nomatch": "eliminate an impossible pattern match",
    "noncomm_ring": "normalize a noncommutative ring expression",
    "nontriviality": "prove the type is nontrivial",
    "norm_cast": "normalize casts in the goal",
    "norm_cast0": "normalize a cast in argument position zero",
    "norm_num": "normalize numeric expressions",
    "norm_num1": "normalize numbers with a smaller lemma set",
    "nth_rewrite": "rewrite at the nth occurrence",
    "nth_rw": "rewrite at the nth occurrence",
    "obtain": "destruct a hypothesis into named parts",
    "omega": "linear integer arithmetic",
    "on_goal": "run a tactic on one goal by index",
    "open": "open a namespace",
    "pderiv_simp": "simplify partial derivatives",
    "peel": "peel off forall or exists binders",
    "pgame_wf_tac": "well-foundedness for progressive games",
    "pi_lower_bound": "lower bound involving pi",
    "pi_upper_bound": "upper bound involving pi",
    "pick_goal": "choose which goal to work on",
    "positivity": "prove positivity of an expression",
    "push_cast": "push casts outward",
    "push_neg": "push negations inwards",
    "qify": "cast a proposition into a quasi-ordering",
    "rcases": "introduce with pattern-based destructuring",
    "rcongr": "recursive congruence",
    "refine": "refine the goal with a partial term",
    "refine'": "refine with explicit holes",
    "rel": "prove a relational goal",
    "rename": "rename a hypothesis",
    "rename'": "rename a hypothesis, structured form",
    "rename_i": "rename inaccessible names",
    "repeat": "apply a tactic until it fails",
    "repeat'": "apply a tactic at least once, then repeatedly",
    "replace": "replace a hypothesis with an equivalent one",
    "revert": "turn hypotheses back into the goal",
    "rewrite": "rewrite with an equation",
    "rfl": "close the goal by reflexivity",
    "rify": "cast an expression into a ring",
    "right": "prove a disjunction by its right disjunct",
    "ring": "commutative ring normalization",
    "ring!": "ring normalization with higher transparency",
    "ring1": "ring-normalize a reflexivity goal",
    "ring_nf": "ring normal form, structural",
    "rintro": "introduce with pattern-based destructuring",
    "rotate_left": "rotate the list of open goals",
    "rsuffices": "introduce auxiliary claims with naming",
    "run_tac": "run an arbitrary tactic expression",
    "rw": "rewrite with an equation",
    "rw_mod_cast": "rewrite after normalizing casts",
    "rwa": "rewrite and then close with assumption",
    "set": "name a subexpression as a local definition",
    "set!": "name a subexpression, overwriting an existing name",
    "set_option": "set a Lean option for the goal",
    "show": "state the goal explicitly",
    "simp": "simplify with a lemma set",
    "simp!": "simplify and report the lemmas used",
    "simp?": "suggest a simplification call",
    "simp_all": "simplify the goal and every hypothesis",
    "simp_all!": "simplify everything and report lemmas used",
    "simp_arith": "simplify using arithmetic lemmas",
    "simp_intro": "simplify and then introduce",
    "simp_rw": "simplify and then rewrite",
    "simp_wf": "simplify with well-founded-recursion lemmas",
    "simpa": "simplify and then close with exact",
    "simpa!": "simpa, reporting the lemmas used",
    "simpa?": "suggest a simpa call",
    "slice_lhs": "slice the left-hand side of an equation",
    "slice_rhs": "slice the right-hand side of an equation",
    "solve_by_elim": "search a proof from assumptions and applications",
    "specialize": "specialize a universally quantified hypothesis",
    "split": "split a conjunction or a constructor goal",
    "split_ifs": "split if-then-else conditions",
    "subst": "substitute an equality",
    "subst_hom_lift": "substitute inside a homomorphism lift",
    "subst_vars": "substitute every possible variable",
    "substs": "substitute several equalities",
    "suffices": "introduce an auxiliary claim to prove",
    "swap": "swap the order of two goals",
    "swap_var": "swap two variables",
    "symm": "apply symmetry of an equality or relation",
    "sz_positivity": "positivity for cardinalities and orders",
    "tauto": "tautology prover",
    "tfae_finish": "finish a list of equivalent statements",
    "tfae_have": "add an item to a list of equivalent statements",
    "to_encard_tac": "convert to enumerable cardinality",
    "trans": "one transitivity step",
    "transfer": "transfer a result along an equivalence",
    "transfer_rw": "rewrite by transferring along an equivalence",
    "transitivity": "prove by transitivity with a witness",
    "trivial": "close an easy goal",
    "try": "run a tactic and ignore failure",
    "unfold": "unfold a definition with simplification",
    "unfold_let": "unfold local definitions",
    "unfold_projs": "unfold structure projections",
    "unit_interval": "show a value lies in the unit interval",
    "use": "provide a witness for an existential goal",
    "use!": "provide a witness with instance synthesis",
    "valid": "prove a validity goal",
    "with_unfolding_all": "unfold everything while rewriting",
    "witt_truncateFun_tac": "Witt vector truncation function",
    "wlog": "reduce to the case without loss of generality",
    "zify": "cast goals into integer arithmetic",
}
DEFAULT_GLOSS = "Lean 4 tactic"


# One-line glosses make the option text informative for a general-purpose model.
missing_gloss = sorted(
    set(bundle.tactic_vocab) - set(TACTIC_GLOSS) - {"<UNK_TACTIC>", "<EMPTY_TACTIC>"}
)
assert not missing_gloss, f"missing glosses for: {missing_gloss}"
print(f"glosses: {len(TACTIC_GLOSS)} tactic families covered")


def laya_state(text_state: str) -> str:
    """Goal first, then the local context.

    Laya keeps the *head* of a state that exceeds its token budget, so putting
    the goal up front guarantees the most informative part is never truncated
    by a long local context.
    """
    parsed = core.parse_state(text_state)
    context = "\n".join(f"{h.name} : {h.type_expr}" for h in parsed.hypotheses)
    return f"GOAL: {parsed.goal}\nCONTEXT: {context}"


def choice_question(candidates: list[str]) -> dict:
    return {
        "tactic": {
            "type": "choice",
            "instructions": LAYA_INSTRUCTIONS,
            "criteria": {c: TACTIC_GLOSS.get(c, DEFAULT_GLOSS) for c in candidates},
        }
    }


# --- smoke test -------------------------------------------------------------
laya_smoke_state = laya_state("α : Type\ninst✝ : Group α\nh : α = α\n⊢ α = α")
smoke_candidates = ["rw", "simp", "exact", "apply", "refine"]
smoke_result = agent.predict(laya_smoke_state, choice_question(smoke_candidates))
smoke_answer = smoke_result["answers"]["tactic"]

assert smoke_answer["type"] == "choice"
assert set(smoke_answer["probabilities"]) == set(smoke_candidates)
assert abs(sum(smoke_answer["probabilities"].values()) - 1.0) < 1e-3
print("smoke choice :", smoke_answer["choice"], "| answer_confidence", smoke_answer["answer_confidence"])
print("probabilities :", smoke_answer["probabilities"])
print("usage         :", smoke_result.get("usage"))

In [ ]:
import json
import time

CHECKPOINT_EVERY = 50

candidates = [ranking[:POOL_K] for ranking in gnn_rankings]
assert len(candidates) == len(state_texts) == len(gt_names)
assert all(len(c) == POOL_K for c in candidates)

LAYA_CACHE = CACHE_DIR / f"laya_{SPLIT}_{N_ROWS}_seed{SEED}_k{POOL_K}.parquet"
records: list[dict] = []
if LAYA_CACHE.exists():
    records = pd.read_parquet(LAYA_CACHE).to_dict("records")
    print(f"resuming: {len(records)}/{len(state_texts)} rows already scored")

started = time.perf_counter()
for i in tqdm(range(len(records), len(state_texts)), desc="laya select"):
    cands = candidates[i]
    answer = agent.predict(
        laya_state(state_texts[i]), choice_question(cands)
    )["answers"]["tactic"]
    probs_by_label = answer["probabilities"]
    # System B's ordering of the ten: Laya's calibrated probability, descending.
    order = sorted(cands, key=lambda label: -probs_by_label.get(label, 0.0))
    records.append({
        "i": i,
        "order": json.dumps(order, ensure_ascii=False),
        "probs": json.dumps([probs_by_label.get(label, 0.0) for label in order]),
        "choice": answer["choice"],
        "answer_confidence": float(answer["answer_confidence"]),
    })
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(records).to_parquet(LAYA_CACHE, index=False)

laya_df = pd.DataFrame(records).sort_values("i").reset_index(drop=True)
laya_df.to_parquet(LAYA_CACHE, index=False)
elapsed = time.perf_counter() - started
if elapsed > 0 and len(records):
    print(f"scored {len(records)} rows in {elapsed:.1f}s "
          f"({1000 * elapsed / len(records):.1f} ms/call) -> {LAYA_CACHE}")

laya_rankings = [json.loads(text) for text in laya_df["order"]]
laya_prob_rows = [json.loads(text) for text in laya_df["probs"]]
assert len(laya_rankings) == len(gnn_rankings)
assert all(sorted(order) == sorted(cands) for order, cands in zip(laya_rankings, candidates))
print("every row's Laya ranking is a permutation of that row's GNN top-10: OK")
print("mean answer confidence:", round(float(laya_df["answer_confidence"].mean()), 4))

# System B's ordering of the pool and its two cut-offs.
known_idx = [i for i, keep in enumerate(known_mask) if keep]
in_pool = [i for i in known_idx if candidate_mask[i]]
best5 = float(np.mean([gt_names[i] in laya_rankings[i][:SELECT_K] for i in known_idx]))
cond5 = float(np.mean([gt_names[i] in laya_rankings[i][:SELECT_K] for i in in_pool]))
print(f"\nSystem B (Laya) — advanced top-{SELECT_K} out of the top-{POOL_K} pool:")
print(f"   over all {len(known_idx)} known rows   {best5:.4f}")
print(f"   given target in the pool ({len(in_pool)})  {cond5:.4f}")

## 7. The two systems, two cut-offs

System A is the pool exactly as the GNN produced it: the ten candidates stay in descending
`p_gnn` order. System B hands the same ten to Laya and keeps them in descending `p_laya`
order. From either ordering the system advances its first **five**, and the first of those
five is the final pick — `recall@5` and `recall@1` are two cut-offs of the same ranking, not
two separate stages.

Because both systems reorder the same ten, they share one pool ceiling:

| system | ordering of the pool | ceiling |
|---|---|---|
| `A — pure GNN`   | `p_gnn` over the ten, unchanged | `recall@10` |
| `B — GNN + Laya` | Laya's calibrated `choice` probabilities over the ten | `recall@10` |

Reported for each system:

* `recall@5` — **the headline**: how often the target survives into the advanced five;
* `recall@1` — how often the target is the final pick;
* `recall@10` — the shared pool ceiling, repeated (a bound, not a result);
* `cond@5` / `cond@1` — the same two numbers restricted to rows whose target was among the ten,
  i.e. the regime where selecting can actually help;
* `mrr@10` — mean reciprocal rank of the target within the system's ranking;
* transitions at `@1` and `@5` — relative to the pure GNN, how many mistakes Laya
  **corrected**, how many correct answers it **broke**, and how many both got right.


In [ ]:
# --- masks (pool membership was computed when the pool was cut) --------------
n = len(gt_names)
rerank_mask = [known_mask[i] and candidate_mask[i] for i in range(n)]
print(f"known targets    : {sum(known_mask)}")
print(f"target in top-{POOL_K}   : {sum(rerank_mask)}  "
      f"({100 * sum(rerank_mask) / max(sum(known_mask), 1):.1f}% of known — the shared ceiling)")

# --- metrics for the two systems --------------------------------------------
SYSTEMS = [
    ("A — pure GNN", "gnn", gnn_rankings),
    ("B — GNN + Laya", "laya", laya_rankings),
]

rows = []
for label, key, rankings in SYSTEMS:
    overall = evaluate(rankings, known_mask)
    conditional = evaluate(rankings, rerank_mask)
    rows.append({
        "system": label,
        "pool": POOL_K,
        "advance": SELECT_K,
        "n": overall["n"],
        "recall@10": overall.get(f"recall@{TOP_K[0]}"),
        f"recall@{SELECT_K}": overall.get(f"recall@{SELECT_K}"),
        "recall@1": overall.get("recall@1"),
        "cond@5": conditional.get("recall@5"),
        "cond@1": conditional.get("recall@1"),
        f"mrr@{TOP_K[0]}": overall.get(f"mrr@{TOP_K[0]}"),
    })

metrics_df = pd.DataFrame(rows)

# The pool ceiling is the same number for both systems: they reorder the same ten.
ceilings = metrics_df[f"recall@{TOP_K[0]}"].to_numpy()
assert np.ptp(ceilings) < 1e-12, f"systems disagree on the pool ceiling: {ceilings}"


def show(frame: pd.DataFrame, decimals: int = 4) -> None:
    # Pretty-print a frame, falling back to plain text if jinja2 is missing.
    try:
        fmt = {c: "{:.%df}" % decimals for c in frame.columns
               if frame[c].dtype.kind in "fc"}
        display(frame.style.format(fmt).hide(axis="index"))
    except Exception:
        print(frame.to_string(index=False))


show(metrics_df)

print(f"\nheadline — recall@{SELECT_K} (the advanced five) and recall@1 (the final pick)")
for _, row in metrics_df.iterrows():
    print(f"   {row['system']:<16} recall@{SELECT_K} = {row[f'recall@{SELECT_K}']:.4f}"
          f"   recall@1 = {row['recall@1']:.4f}"
          f"   (cond@5 {row['cond@5']:.4f}, ceiling {row['recall@10']:.4f})")


# --- what Laya changed relative to the pure GNN, at both cut-offs -----------
def transitions(reference, candidate, mask, *, k):
    idx = [i for i, keep in enumerate(mask) if keep]
    ref_ok = [gt_names[i] in reference[i][:k] for i in idx]
    cand_ok = [gt_names[i] in candidate[i][:k] for i in idx]
    total = len(idx) or 1
    return {
        "both right": sum(a and b for a, b in zip(ref_ok, cand_ok)) / total,
        "only GNN right": sum(a and not b for a, b in zip(ref_ok, cand_ok)) / total,
        "only Laya right": sum((not a) and b for a, b in zip(ref_ok, cand_ok)) / total,
        "neither": sum((not a) and (not b) for a, b in zip(ref_ok, cand_ok)) / total,
        "n": len(idx),
    }


transitions_df = pd.DataFrame({
    "top-1": transitions(gnn_rankings, laya_rankings, known_mask, k=1),
    f"top-{SELECT_K}": transitions(gnn_rankings, laya_rankings, known_mask, k=SELECT_K),
}).T
show(transitions_df)


## 8. Plots

Four figures:

1. **Recall at each cut-off** for the two systems (`recall@10`, `recall@5`, `recall@1`), with
   the shared pool ceiling marked.
2. **Recall@5 — the advanced five** — for each system, both over all rows and restricted to
   rows whose target was in the pool.
3. **What Laya changed** — at `top-1` and at `top-5`: how many GNN mistakes it corrected, how
   many correct answers it broke, and how many both agreed on.
4. **Where the target ended up** — the target's rank under the pure GNN against its rank under
   Laya, within the pool (bucket `>10` = outside the pool).


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.axisbelow": True,
    "font.size": 10,
})

SYSTEM_LABELS = [label for label, _, _ in SYSTEMS]
GNN_LABEL, LAYA_LABEL = SYSTEM_LABELS
COLORS = {GNN_LABEL: "#4C72B0", LAYA_LABEL: "#DD8452"}
CEILING = float(metrics_df.loc[metrics_df["system"] == GNN_LABEL,
                               f"recall@{TOP_K[0]}"].iloc[0])

# ---------------------------------------------------------------- figure 1 --
fig, ax = plt.subplots(figsize=(7.4, 4.2))
ks = list(TOP_K)
width = 0.36
x = np.arange(len(ks))
for offset, label in zip((-0.5, 0.5), SYSTEM_LABELS):
    row = metrics_df.loc[metrics_df["system"] == label].iloc[0]
    values = [row[f"recall@{k}"] for k in ks]
    bars = ax.bar(x + offset * width, values, width, label=label, color=COLORS[label])
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)

ax.axhline(CEILING, ls="--", lw=1.2, color="black", alpha=0.6)
ax.text(len(ks) - 0.55, CEILING + 0.012,
        f"pool ceiling {CEILING:.3f}", ha="right", fontsize=8.5)

ax.set_xticks(x, [f"recall@{k}" for k in ks])
ax.set_ylabel("fraction of rows with the target in the ranking")
ax.set_ylim(0, 1.06)
ax.set_title(f"Two systems over the GNN's top-{POOL_K} pool")
ax.legend(ncols=2, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12))
fig.tight_layout()
fig.savefig(OUT_DIR / "fig1_recall.png", bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------- figure 2 --
# The headline: how often the target survived into each system's advanced five.
fig, ax = plt.subplots(figsize=(7.0, 4.2))
idx = np.arange(len(SYSTEM_LABELS))
overall5 = [float(metrics_df.loc[metrics_df["system"] == s,
                                 f"recall@{SELECT_K}"].iloc[0]) for s in SYSTEM_LABELS]
cond5 = [float(metrics_df.loc[metrics_df["system"] == s, "cond@5"].iloc[0])
         for s in SYSTEM_LABELS]

b1 = ax.bar(idx - 0.19, overall5, 0.36, label=f"all {sum(known_mask)} known rows",
            color=[COLORS[s] for s in SYSTEM_LABELS])
b2 = ax.bar(idx + 0.19, cond5, 0.36, label="given the target is in the pool",
            color="#BBBBBB")
ax.bar_label(b1, fmt="%.3f", padding=2, fontsize=9)
ax.bar_label(b2, fmt="%.3f", padding=2, fontsize=9, color="#666666")
ax.axhline(CEILING, ls="--", lw=1.4, color="black", alpha=0.6)
ax.text(len(SYSTEM_LABELS) - 0.42, CEILING + 0.015,
        f"ceiling {CEILING:.3f}", ha="right", fontsize=9)

ax.set_xticks(idx, SYSTEM_LABELS)
ax.set_ylabel(f"fraction of rows with the target in the advanced top-{SELECT_K}")
ax.set_ylim(0, 1.08)
ax.set_title(f"Recall@{SELECT_K} — the five each system advances")
ax.legend(ncols=2, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12),
          fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig2_top5.png", bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------- figure 3 --
fig, ax = plt.subplots(figsize=(7.0, 4.2))
stack_cols = ["both right", "only GNN right", "only Laya right", "neither"]
stack_colors = ["#55A868", "#4C72B0", "#DD8452", "#BBBBBB"]
stack_rows = list(transitions_df.index)
bottom = np.zeros(len(stack_rows))
for column, colour in zip(stack_cols, stack_colors):
    values = transitions_df.loc[stack_rows, column].to_numpy(dtype=float)
    ax.bar(stack_rows, values, bottom=bottom, color=colour, label=column, width=0.55)
    for k, (value, base) in enumerate(zip(values, bottom)):
        if value > 0.03:
            ax.text(k, base + value / 2, f"{value:.3f}", ha="center", va="center",
                    fontsize=8.5, color="white" if colour != "#BBBBBB" else "black")
    bottom += values

ax.set_ylabel("share of rows")
ax.set_ylim(0, 1.02)
ax.set_title("What Laya changed relative to the pure GNN, at both cut-offs")
ax.legend(ncols=2, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.10))
fig.tight_layout()
fig.savefig(OUT_DIR / "fig3_transitions.png", bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------- figure 4 --
MAX_RANK = POOL_K + 1   # bucket 11 = "not in the pool"


def capped_rank(ranking, index):
    r = rank_of(gt_names[index], ranking)
    return r if (r is not None and r <= POOL_K) else MAX_RANK


pair_grid = np.zeros((MAX_RANK, MAX_RANK), dtype=int)
for i, keep in enumerate(known_mask):
    if keep:
        pair_grid[capped_rank(laya_rankings, i) - 1, capped_rank(gnn_rankings, i) - 1] += 1

fig, ax = plt.subplots(figsize=(5.6, 4.8))
im = ax.imshow(pair_grid, cmap="Greens")
labels = [str(k) for k in range(1, POOL_K + 1)] + [f">{POOL_K}"]
ax.set_xticks(range(MAX_RANK), labels)
ax.set_yticks(range(MAX_RANK), labels)
ax.set_xlabel("rank of the target under the pure GNN")
ax.set_ylabel("rank of the target under Laya")
ax.set_title("Per-row target rank: pure GNN (x) vs Laya (y)")
for a in range(MAX_RANK):
    for b in range(MAX_RANK):
        value = pair_grid[a, b]
        if value:
            ax.text(b, a, str(value), ha="center", va="center", fontsize=7,
                    color="white" if value > pair_grid.max() * 0.55 else "black")
ax.plot([0, MAX_RANK - 1], [0, MAX_RANK - 1], ls="--", lw=1, color="black", alpha=0.4)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="rows")
fig.tight_layout()
fig.savefig(OUT_DIR / "fig4_rank_shift.png", bbox_inches="tight")
plt.show()

print("saved:", *[p.name for p in sorted(OUT_DIR.glob("fig*.png"))], sep="\n  ")


In [ ]:
theorem_by_row = eval_df.set_index("row_index")["theorem"].to_dict()

records = []
for i in range(len(gt_names)):
    row_index = int(graphs[i].row_index)
    known = known_mask[i]
    rec = {
        "row_index": row_index,
        "theorem": theorem_by_row.get(row_index, ""),
        "target": gt_names[i],
        "target_known": known,
        f"target_in_top{POOL_K}": bool(candidate_mask[i]),
        "gnn_top1_prob": round(float(probs[i, bundle.tactic_vocab[gnn_rankings[i][0]]]), 6),
        "laya_top1_prob": round(float(laya_prob_rows[i][0]), 6),
        "laya_answer_confidence": round(float(laya_df.loc[i, "answer_confidence"]), 4),
    }
    for rank, label in enumerate(gnn_rankings[i], start=1):
        rec[f"gnn_{rank}"] = label
    for rank, label in enumerate(laya_rankings[i], start=1):
        rec[f"laya_{rank}"] = label
        rec[f"laya_p_{rank}"] = round(float(laya_prob_rows[i][rank - 1]), 4)
    for label, key, rankings in SYSTEMS:
        rec[f"{key}_ok"] = bool(known and rankings[i][0] == gt_names[i])
        rec[f"{key}_top{SELECT_K}_ok"] = bool(
            known and gt_names[i] in rankings[i][:SELECT_K]
        )
    records.append(rec)

per_row = pd.DataFrame(records)
per_row_path = OUT_DIR / "per_row_rankings.csv"
per_row.to_csv(per_row_path, index=False)

metrics_path = OUT_DIR / "metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

# ---------------------------------------------------------------- summary ----
def _fmt(value):
    return f"{value:.4f}" if isinstance(value, float) else str(value)


lines = [
    f"# Two systems over a top-{POOL_K} pool — advanced top-{SELECT_K} and final top-1",
    "",
    f"* model bundle : `{MODEL_REPO}` / `{BUNDLE_NAME}`",
    f"* dataset      : `{DATASET_REPO}` split `{SPLIT}`",
    f"* rows         : {len(gt_names)} (seed {SEED}); {sum(known_mask)} with a known target",
    f"* pool         : the GNN's top-{POOL_K}; system A keeps that order, "
    f"system B re-ranks the same ten with Laya",
    f"* cut-offs     : each system advances its top-{SELECT_K}; its #1 is the final pick",
    "",
    "## Metrics",
    "",
]
lines.append("| " + " | ".join(metrics_df.columns) + " |")
lines.append("|" + "---|" * len(metrics_df.columns))
for _, row in metrics_df.iterrows():
    lines.append("| " + " | ".join(_fmt(v) for v in row.tolist()) + " |")

lines += [
    "",
    "## What Laya changed relative to the pure GNN (share of rows)",
    "",
]
lines.append("| " + " | ".join(["cut-off"] + list(transitions_df.columns)) + " |")
lines.append("|" + "---|" * (len(transitions_df.columns) + 1))
for name, row in transitions_df.iterrows():
    lines.append("| " + " | ".join([name] + [_fmt(v) for v in row.tolist()]) + " |")

lines += [
    "",
    "## Files",
    "",
    f"* `{per_row_path.name}` — one row per evaluated proof state, both rankings",
    f"* `{metrics_path.name}` — the metrics table above",
    f"* figures: " + ", ".join(p.name for p in sorted(OUT_DIR.glob("fig*.png"))),
    "",
]

summary_path = OUT_DIR / "summary.md"
summary_path.write_text("\n".join(lines), encoding="utf-8")

print("wrote:")
for path in (per_row_path, metrics_path, summary_path):
    print(f"  {path}  ({path.stat().st_size / 1024:.1f} kB)")
print("\n" + "\n".join(lines[:40]))


## 9. How to read the results — and what they do *not* show

**`recall@5` is the result.** Both systems receive the same ten candidates and each advances
five, so `recall@5` — and its conditional twin `cond@5`, restricted to rows whose target was in
the pool at all — is the number the comparison turns on. `recall@10` is the shared pool ceiling
repeated twice: a bound, not a result. It is asserted equal across the two systems, because
they only ever reorder the same ten.

**`recall@1` is the final pick**, the head of the advanced five. The transition table and
figure 3 show what Laya changed relative to the pure GNN at *both* cut-offs — a re-ranker can
gain at `@5` while losing at `@1` if it breaks more correct heads than it fixes.

**`cond@5` / `cond@1` are the fairer comparison.** Roughly half the rows have a target the GNN
never put in the pool; no system can fix that, so those rows dilute the overall numbers equally
for both systems. The conditional columns remove them and show only the rows where selecting
could make a difference.

**Laya only ever sees the ten, on purpose.** A `choice` question spends one shared
`head_max_len` (192 tokens) on every option and refuses the question once they stop fitting —
about 20 options. The vocabulary has 226 usable tactics, so "Laya picks from all of them"
cannot be asked without an option budget several times the checkpoint's trained context.
`predict_shortlist` would work around it with an encoder cosine shortlist, but then the first
cut would not be Laya's. Laya is therefore evaluated where it is sound: as a re-ranker of the
GNN's pool.

**Laya is a general decision model, not a Lean expert.** Its only contact with the domain is the
one-line gloss attached to each option. It has no notion of the proof state's type theory beyond
what the English rendering conveys, and `ModernBERT-large` reads a right-truncated window of
`max_len - head_max_len` ≈ 320 tokens. Treat it as a cheap, calibration-aware *re-ranker of a
shortlist*, which is exactly the setup evaluated here.

**Scope.** This notebook evaluates the *tactic family* ranking only. The published bundle also
contains an argument-pointer head (`argument_selector` / `stop_head` / `tactic_embedding`,
dropped at load time) and a premise scorer (`scorer.safetensors`, untouched); neither has a
published headline metric for this dataset, so neither is scored here.

**Reproducibility.** Everything downstream of the sample is deterministic: `SEED` fixes the
sample, graph construction is pure CPU string parsing, the GNN runs in `eval()` mode under
`torch.no_grad()`, and Laya applies its shipped temperature calibration. Rerunning the notebook
reuses both caches (`cache/graphs_*.pt`, `cache/laya_*.parquet`); delete them to start over.
